<a href="https://colab.research.google.com/github/aadityane93/Toxic_Comments_Sentiment_Analysis/blob/aaditya/4_Multi_Label_Tranfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import

In [ ]:
# !pip install datasets transformers datasets scikit-learn #For Colab

In [ ]:
## Checking if my laptop GPU works
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU found")

GPU available: True
GPU name: NVIDIA GeForce RTX 5060 Laptop GPU


In [ ]:
# Basic libraries
import pandas as pd
import numpy as np
import torch
import inspect

# Hugging Face libraries
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ML utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

C:\Users\aaditya\Desktop\Deep Learning Project\project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data

In [ ]:
# Load CSV files
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
test_labels = pd.read_csv("test_labels.csv")

# Show data shape
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Show first rows
train_df.head()

# These are the six output labels for multi-label classification
target_cols = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

Train shape: (159571, 8)
Test shape: (153164, 2)


## Split training and validation data

In [ ]:
# Split original training data into training and validation sets
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42
)

# Reset index
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

print("Training data:", train_data.shape)
print("Validation data:", val_data.shape)

Training data: (127656, 8)
Validation data: (31915, 8)


## Convert pandas DataFrame to Hugginf Face Dataset

In [ ]:
# Convert pandas DataFrame to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

print(train_dataset.column_names)

['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


## Load DistilBERT tokenizer

In [ ]:
# DistilBERT model name
model_name = "distilbert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

## Tokenize text and prepare labels

In [ ]:
# this function tokenize the comment and create one "label" column that contains all 6 labels
def preprocess_data(batch):

    # Convert text into tokens
    encoding = tokenizer(
        batch["comment_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # Create labels for each comment
    labels = []

    # Loop through each comment in the batch
    for i in range(len(batch["comment_text"])):

        one_comment_labels = []
        # Collect labels: toxic, severe_toxic, obscene, etc.
        for col in target_cols:
            one_comment_labels.append(float(batch[col][i]))

        # Add this comment's labels
        labels.append(one_comment_labels)

    # Add labels to the encoded data
    encoding["labels"] = labels

    return encoding

## Apply preprocessing

In [ ]:
# Tokenize training dataset and remove old columns
train_dataset = train_dataset.map(
    preprocess_data,
    batched=True,
    remove_columns=train_dataset.column_names
)

# Tokenize validation dataset and remove old columns
val_dataset = val_dataset.map(
    preprocess_data,
    batched=True,
    remove_columns=val_dataset.column_names
)

Map: 100%|█████████████████████████████████████████████████████████████| 31915/31915 [00:01<00:00, 16269.58 examples/s]


In [ ]:
# After preprocessing, we should only have these columns:
print(train_dataset.column_names)
print(val_dataset.column_names)

['input_ids', 'token_type_ids', 'attention_mask', 'labels']
['input_ids', 'token_type_ids', 'attention_mask', 'labels']


## Set dataset format for PyTorch

In [ ]:
# Convert dataset columns to PyTorch tensors
train_dataset.set_format("torch")
val_dataset.set_format("torch")

## Load pretrained DistilBERT model

In [ ]:
# Label ID to label name
id2label = {
    0: "toxic",
    1: "severe_toxic",
    2: "obscene",
    3: "threat",
    4: "insult",
    5: "identity_hate"
}

# Label name to label ID
label2id = {
    "toxic": 0,
    "severe_toxic": 1,
    "obscene": 2,
    "threat": 3,
    "insult": 4,
    "identity_hate": 5
}

# Load pretrained DistilBERT for multi-label classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

C:\Users\aaditya\Desktop\Deep Learning Project\project\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aaditya\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|█████████████████████████████████████████████████████████████|

## Define evaluation metrics

In [ ]:
def compute_metrics(eval_pred):

    #This function calculates:
   # 1. ROC-AUC
   # 2. Precision
   # 3. Recall
   # 4. F1-score

    # Get model outputs and true labels
    logits, labels = eval_pred

    # Convert logits to probabilities using sigmoid
    probabilities = 1 / (1 + np.exp(-logits))

    # Convert probabilities to 0 or 1 using threshold 0.5
    predictions = (probabilities >= 0.5).astype(int)

    # ROC-AUC
    roc_auc = roc_auc_score(
        labels,
        probabilities,
        average="macro"
    )

    # Precision
    precision = precision_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    # Recall
    recall = recall_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    # F1-score
    f1 = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "roc_auc": roc_auc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

## Define training settings

In [ ]:
# Basic training settings
args_dict = {
    "output_dir": "./distilbert-toxic-model",

    # Save model after each epoch
    "save_strategy": "epoch",

    # Train for 2 epochs
    "num_train_epochs": 2,

    # Batch size
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 16,

    # Learning rate
    "learning_rate": 2e-5,

    # Regularization
    "weight_decay": 0.01,

    # Load best model after training
    "load_best_model_at_end": True,

    # Use ROC-AUC to choose best model
    "metric_for_best_model": "roc_auc",
    "greater_is_better": True,

    # Disable wandb
    "report_to": "none"
}

training_args_params = inspect.signature(TrainingArguments.__init__).parameters

if "eval_strategy" in training_args_params:
    args_dict["eval_strategy"] = "epoch"
else:
    args_dict["evaluation_strategy"] = "epoch"


# Create Training Arguments object
training_args = TrainingArguments(
    output_dir="./distilbert-toxic-model",

    eval_strategy="epoch",
    save_strategy="epoch",

    num_train_epochs=2,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    learning_rate=2e-5,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    greater_is_better=True,

    report_to="none",

    # Use mixed precision only if GPU is available
    fp16=torch.cuda.is_available()
)

## Create Trainer

In [ ]:
# Create Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Roc Auc,Precision,Recall,F1
1,0.015591,0.056820,0.987000,0.705936,0.620067,0.654961
2,0.016262,0.058574,0.987842,0.688146,0.656991,0.670172


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.20it/s]


TrainOutput(global_step=15958, training_loss=0.016226267393317523, metrics={'train_runtime': 751.0968, 'train_samples_per_second': 339.919, 'train_steps_per_second': 21.246, 'total_flos': 8455732262313984.0, 'train_loss': 0.016226267393317523, 'epoch': 2.0})

## Evaluate on validation data

In [ ]:
# Evaluate on validation set
val_results = trainer.evaluate()

# Print results
print(val_results)

Training Loss,Validation Loss,Epoch,Roc Auc,Precision,Recall,F1
0.016262,0.058574,2,0.987842,0.688146,0.656991,0.670172


{'eval_loss': 0.05857379361987114, 'eval_roc_auc': 0.9878419620961792, 'eval_precision': 0.6881462677227694, 'eval_recall': 0.6569905899222473, 'eval_f1': 0.6701722883354785}


In [ ]:
# Merge test comments with test labels
test_full = test_df.merge(test_labels, on="id")

# Remove rows where all labels are -1
# -1 means those rows were not used for scoring
test_full = test_full[test_full[target_cols].sum(axis=1) != -6]

# Reset index
test_full = test_full.reset_index(drop=True)

print("Usable test data:", test_full.shape)

Usable test data: (63978, 8)


In [ ]:
# Convert test DataFrame to Hugging Face Dataset
test_dataset = Dataset.from_pandas(test_full)

# Apply same preprocessing
test_dataset = test_dataset.map(
    preprocess_data,
    batched=True,
    remove_columns=test_dataset.column_names
)

# Convert to PyTorch format
test_dataset.set_format("torch")

print(test_dataset.column_names)

Map: 100%|█████████████████████████████████████████████████████████████| 63978/63978 [00:03<00:00, 17094.25 examples/s]

['input_ids', 'token_type_ids', 'attention_mask', 'labels']


## Evaluation on Test Data

In [ ]:
# Evaluate on test dataset
test_results = trainer.evaluate(
    eval_dataset=test_dataset
)

# Print test results
print(test_results)

Training Loss,Validation Loss,Epoch,Roc Auc,Precision,Recall,F1
0.016262,0.110514,2,0.979601,0.551836,0.666563,0.597544


{'eval_loss': 0.1105135902762413, 'eval_roc_auc': 0.9796013484629205, 'eval_precision': 0.5518362978299848, 'eval_recall': 0.6665625695206672, 'eval_f1': 0.5975440127605075}


## On 5 epoch
got 0.006935	0.072189	5	0.981189	0.582486	0.606722	0.587985

## Validation Outputs for Model Evaluation

In [ ]:

# Get model predictions on validation data
val_output = trainer.predict(val_dataset)

# Trainer.predict returns raw logits and labels
# Hugging Face Trainer predict returns predictions and label_ids
val_logits = val_output.predictions
val_labels = val_output.label_ids

# Convert logits to probabilities using sigmoid
val_probs = 1 / (1 + np.exp(-val_logits))

## Label wise Threshold Optimization for Best F1 Score

In [ ]:
best_thresholds = []

for i, label in enumerate(target_cols):

    best_f1 = 0
    best_threshold = 0.5

    # Trying the   thresholds from 0.05 to 0.95
    for threshold in np.arange(0.05, 0.96, 0.01):

        # Convert probabilities to 0/1
        preds = (val_probs[:, i] >= threshold).astype(int)

        # Calculate F1 for this label
        f1 = f1_score(
            val_labels[:, i],
            preds,
            zero_division=0
        )

        # Save threshold if F1 improves
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    best_thresholds.append(best_threshold)

    print(label)
    print("Best threshold:", round(best_threshold, 2))
    print("Best validation F1:", round(best_f1, 4))
    print()

toxic
Best threshold: 0.45
Best validation F1: 0.825

severe_toxic
Best threshold: 0.21
Best validation F1: 0.5236

obscene
Best threshold: 0.32
Best validation F1: 0.8294

threat
Best threshold: 0.54
Best validation F1: 0.5921

insult
Best threshold: 0.32
Best validation F1: 0.7659

identity_hate
Best threshold: 0.46
Best validation F1: 0.5766



## Applying Optimized Thresholds to Test Predictions

In [ ]:
# Get test predictions
test_output = trainer.predict(test_dataset)

test_logits = test_output.predictions
test_labels_np = test_output.label_ids

# Convert logits to probabilities
test_probs = 1 / (1 + np.exp(-test_logits))

# Create empty prediction array
test_preds = np.zeros_like(test_probs)

# Apply different threshold for each label
for i, threshold in enumerate(best_thresholds):
    test_preds[:, i] = (test_probs[:, i] >= threshold).astype(int)

## Final Test Metrics Calculating

In [ ]:
test_roc_auc = roc_auc_score(
    test_labels_np,
    test_probs,
    average="macro"
)

test_precision = precision_score(
    test_labels_np,
    test_preds,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    test_labels_np,
    test_preds,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    test_labels_np,
    test_preds,
    average="macro",
    zero_division=0
)

print("Test ROC-AUC:", test_roc_auc)
print("Test Precision:", test_precision)
print("Test Recall:", test_recall)
print("Test F1:", test_f1)

Test ROC-AUC: 0.9796013484629205
Test Precision: 0.5234890744284567
Test Recall: 0.7144228869542452
Test F1: 0.5914246082544339


## Final Results DataFrame for Each Label

In [ ]:
results = []

for i, label in enumerate(target_cols):

    precision = precision_score(
        test_labels_np[:, i],
        test_preds[:, i],
        zero_division=0
    )

    recall = recall_score(
        test_labels_np[:, i],
        test_preds[:, i],
        zero_division=0
    )

    f1 = f1_score(
        test_labels_np[:, i],
        test_preds[:, i],
        zero_division=0
    )

    auc = roc_auc_score(
        test_labels_np[:, i],
        test_probs[:, i]
    )

    results.append({
        "label": label,
        "threshold": best_thresholds[i],
        "auc": auc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

results_df = pd.DataFrame(results)
results_df

,label,threshold,auc,precision,recall,f1
0,toxic,0.45,0.964931,0.510619,0.908046,0.653664
1,severe_toxic,0.21,0.987723,0.268770,0.673025,0.384137
2,obscene,0.32,0.974608,0.591505,0.807369,0.682782
3,threat,0.54,0.994473,0.535088,0.578199,0.555809
4,insult,0.32,0.971068,0.620477,0.735629,0.673164
5,identity_hate,0.46,0.984805,0.614476,0.584270,0.598992
